In [ ]:
# Import Required Libraries
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
    ExtraTreesClassifier,
)
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold

import os
print("Loading Kaggle input files...")
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [ ]:
# Load Data
print("\nLoading training and test data...")
train_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")

In [ ]:
# Preprocessing Function
def preprocess_data(df, is_training=True, encoder=None, scaler=None, numeric_means=None, target_col='Churn'):
    """
    Preprocess customer churn data with consistent encoding for train/test sets.
    """
    df_processed = df.copy()
    
    # Identify feature columns (exclude id and target)
    feature_cols = [col for col in df_processed.columns if col not in ['id', target_col]]
    
    # Separate numeric and categorical features
    numeric_cols = df_processed[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df_processed[feature_cols].select_dtypes(include=['object']).columns.tolist()
    
    # Handle categorical variables with one-hot encoding
    if is_training:
        df_dummies = pd.get_dummies(df_processed[categorical_cols], drop_first=True)
        encoder = {col: df_processed[col].unique() for col in categorical_cols}
    else:
        df_dummies = pd.get_dummies(df_processed[categorical_cols], drop_first=True)
    
    # Combine numeric and encoded categorical features
    X = pd.concat([df_processed[numeric_cols].reset_index(drop=True), 
                   df_dummies.reset_index(drop=True)], axis=1)
    
    # Handle missing values and scaling
    if is_training:
        numeric_means = X[numeric_cols].mean()
        X[numeric_cols] = X[numeric_cols].fillna(numeric_means)
        scaler = StandardScaler()
        X[numeric_cols] = scaler.fit_transform(X[numeric_cols])
    else:
        if numeric_cols:
            X[numeric_cols] = X[numeric_cols].fillna(numeric_means)
            X[numeric_cols] = scaler.transform(X[numeric_cols])
    
    if is_training:
        y = df_processed[target_col]
        return X, y, encoder, scaler, numeric_means
    else:
        return X, encoder, scaler

# Apply preprocessing to training data
print("\nPreprocessing training data...")
X_train, y_train, encoder, scaler, numeric_means = preprocess_data(train_df, is_training=True)
print(f"Training data shape: {X_train.shape}")
print(f"Target distribution:\n{y_train.value_counts()}")

# Apply preprocessing to test data
print("\nPreprocessing test data...")
X_test, _, _ = preprocess_data(test_df, is_training=False, encoder=encoder, scaler=scaler, numeric_means=numeric_means)

# Ensure test data has same columns as training data
missing_cols = set(X_train.columns) - set(X_test.columns)
extra_cols = set(X_test.columns) - set(X_train.columns)

if missing_cols:
    for col in missing_cols:
        X_test[col] = 0

if extra_cols:
    X_test = X_test.drop(columns=extra_cols)

X_test = X_test[X_train.columns]
print(f"Final test data shape: {X_test.shape}")


In [ ]:
# Ready for model training
print("\nData preprocessing complete.")
print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")

In [ ]:
# Train Stacking Ensemble
print("\nTraining stacking ensemble with logistic regression meta-learner...")
print("="*70)

# Define base models
base_models = [
    ('HistGradientBoosting', HistGradientBoostingClassifier(
        learning_rate=0.15, max_depth=7, max_leaf_nodes=50,
        min_samples_leaf=20, l2_regularization=0.1, max_bins=128, random_state=42,
        max_iter=2000
    )),
    ('RandomForest', RandomForestClassifier(
        n_estimators=1000, max_depth=10, min_samples_leaf=10, random_state=42, n_jobs=-1
    )),
    ('ExtraTrees', ExtraTreesClassifier(
        n_estimators=1000, max_depth=10, min_samples_leaf=10, random_state=42, n_jobs=-1
    )),
    ('LogisticRegression', LogisticRegression(
        C=1.0, max_iter=1000, random_state=42, n_jobs=-1
    )),
    ('MLP', MLPClassifier(
        hidden_layer_sizes=(128, 64), max_iter=300, learning_rate_init=0.001,
        early_stopping=True, validation_fraction=0.1, random_state=42
    )),
]

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

X_train_arr = X_train.values
X_test_arr  = X_test.values
# Encode target as binary integers (0/1)
y_train_arr = (y_train == 'Yes').astype(int).values

# Out-of-fold and test predictions for level-1 features
oof_preds  = np.zeros((len(X_train_arr), len(base_models)))
test_preds = np.zeros((len(X_test_arr),  len(base_models)))

from sklearn.metrics import roc_auc_score

for model_idx, (model_name, base_model) in enumerate(base_models):
    print(f"\nTraining base model: {model_name}")
    fold_test_preds = np.zeros((len(X_test_arr), N_FOLDS))

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_train_arr, y_train_arr)):
        X_fold_train, X_fold_val = X_train_arr[train_idx], X_train_arr[val_idx]
        y_fold_train = y_train_arr[train_idx]

        base_model.fit(X_fold_train, y_fold_train)
        oof_preds[val_idx, model_idx] = base_model.predict_proba(X_fold_val)[:, 1]
        fold_test_preds[:, fold_idx]  = base_model.predict_proba(X_test_arr)[:, 1]
        print(f"  Fold {fold_idx + 1}/{N_FOLDS} complete")

    test_preds[:, model_idx] = fold_test_preds.mean(axis=1)
    oof_auc = roc_auc_score(y_train_arr, oof_preds[:, model_idx])
    print(f"  {model_name} OOF AUC: {oof_auc:.5f}")

print(f"\nOOF predictions shape:  {oof_preds.shape}")
print(f"Test predictions shape: {test_preds.shape}")


In [ ]:

# Train logistic regression meta-learner on OOF predictions
print("\nFitting logistic regression meta-learner...")
meta_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
meta_model.fit(oof_preds, y_train_arr)

# OOF AUC of the full stacked ensemble
stacked_oof_proba = meta_model.predict_proba(oof_preds)[:, 1]
stacked_oof_auc = roc_auc_score(y_train_arr, stacked_oof_proba)
print(f"\nStacked ensemble OOF AUC: {stacked_oof_auc:.5f}")

print("\nMeta-model coefficients:")
for (name, _), coef in zip(base_models, meta_model.coef_[0]):
    print(f"  {name}: {coef:.6f}")
print(f"  Intercept: {meta_model.intercept_[0]:.6f}")
print("="*70)



In [ ]:
# Generate Stacked Predictions
print("\nGenerating stacked predictions via meta-learner...")
print("="*70)

churn_probability = meta_model.predict_proba(test_preds)[:, 1]

print(f"Predictions shape: {churn_probability.shape}")
print(f"Probability range: [{churn_probability.min():.6f}, {churn_probability.max():.6f}]")
print(f"Probability mean:  {churn_probability.mean():.6f}")
print(f"Probability std:   {churn_probability.std():.6f}")
print("="*70)


In [ ]:
# Create and Save Submission File
print("\n" + "="*70)
print("CREATING SUBMISSION FILE")
print("="*70)

# Create submission DataFrame
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'Churn': churn_probability
})

print(f"Submission DataFrame created with shape: {submission_df.shape}")

# Kaggle output directory
import os
output_path = '/kaggle/working/submission.csv'

# Save submission file
try:
    submission_df.to_csv(output_path, index=False)
    print(f"\n✓ File saved to: {output_path}")
    if os.path.exists(output_path):
        file_size = os.path.getsize(output_path)
        print(f"  File size = {file_size / 1024:.2f} KB")
    else:
        print("  ERROR: File not found after save!")
except Exception as e:
    print(f"ERROR saving file: {e}")
    # Fallback to /tmp
    output_path = '/tmp/submission.csv'
    submission_df.to_csv(output_path, index=False)
    print(f"  Saved to fallback location: {output_path}")

print(f"  Number of predictions: {len(submission_df)}")

# Display preview
print(f"\nSubmission Preview (first 15 rows):")
print(submission_df.head(15).to_string())

print("\n" + "="*70)
print("SUBMISSION READY!")
print("="*70)